# Método de Levenberg-Marquardt

Levenberg-Marquardt (LM) se utiliza para resolver problemas de mínimos cuadrados no lineales, minimizando una función objetivo $S(\beta)$ que es la suma de los residuos al cuadrado

\begin{equation}
 S(\beta) = \sum_{i=1}^m [y_i - f(x_i, \beta)]^2 = \Vert{} r(\beta) \Vert{}^2.
\end{equation}

LM es un método "híbrido" que interpola suavemente entre dos algoritmos clásicos mediante un factor de amortiguamiento $\lambda$:

* Si $\lambda$ es grande: Se comporta como el Gradiente Descendente, garantizando que el error converja de forma segura, útil cuando estamos lejos del mínimo.
* Si $\lambda$ es pequeño: Se aproxima al Método de Gauss-Newton, asumiendo que la función es localmente cuadrática para converger con mayor velocidad cerca del mínimo óptimo.

## Algoritmo de LM

1. En cada iteración, se actualiza $\delta$ resolviendo el siguiente sistema lineal

\begin{equation}
 (J^T J + \lambda I) \delta = J^T r.
\end{equation}
donde $J$ es la matriz Jacobiana de la función respecto a los parámetros $\beta$ ($J_{ij} = \frac{\partial f(x_i, \beta)}{\partial \beta_j}$), $r = y - f(x, \beta)$ es el vector de residuos, $I$ es la matriz identidad.

2. Si el nuevo parámetro $\beta_{k+1} = \beta_k + \delta$ disminuye el error, se acepta el paso y se divide $\lambda$ por un factor ($\kappa$). Si el error incrementa, se rechaza la actualización y se multiplica $\lambda$ por $\kappa$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Suprimir advertencias menores de integración durante los bucles de optimización
import warnings
warnings.filterwarnings("ignore")

# Configurar el estilo de graficación para una mejor visualización
plt.style.use('ggplot')

In [2]:
def levenberg_marquardt(x_data, y_data, func, jacobian, beta0, 
                        lambda_init=10.0, tol=1e-8, max_iter=100):
    """Minimiza el error de ajuste no lineal con Levenberg-Marquardt.
    
    Parámetros:
    -----------
    x_data : array-like
        Datos independientes.
    y_data : array-like
        Datos dependientes (observaciones).
    func : callable
        Función modelo func(x_data, beta).
    jacobian: callable
        Función que calcula la matriz Jacobiana J(x_data, beta).
    beta0 : array-like
        Estimación inicial de los parámetros.
    lambda_init : float
        Valor inicial del factor de amortiguamiento.
    tol : float
        Tolerancia para la norma del incremento del paso (delta).
    max_iter : int
        Número máximo de iteraciones permitidas.
        
    Retorna:
    --------
    beta : np.ndarray
        Parámetros optimizados.
    """
    beta = np.array(beta0, dtype=float)
    lam = lambda_init
    
    def compute_error(b):
        return np.sum((y_data - func(x_data, b))**2)
    
    error = compute_error(beta)
    n_params = len(beta)
    
    for i in range(max_iter):
        res = y_data - func(x_data, beta)
        J = jacobian(x_data, beta)
        
        H = J.T @ J
        g = J.T @ res  
        
        # Construcción del sistema amortiguado
        H_lam = H + lam * np.eye(n_params)
        
        try:
            delta = np.linalg.solve(H_lam, g)
        except np.linalg.LinAlgError:
            # Si hay problemas numéricos, incrementamos lambda y reintentamos la iteración
            print(f"Matriz singular o mal condicionada en la iteración {i+1}. Incrementando lambda.")
            lam *= 10.0
            continue
        
        beta_new = beta + delta
        error_new = compute_error(beta_new)
        
        if error_new < error:
            # Paso exitoso: aceptamos el cambio y acercamos LM a Gauss-Newton
            beta = beta_new
            error = error_new
            lam /= 10.0
            
            # Condición de parada
            if np.linalg.norm(delta) < tol:
                print(f"LM convergió exitosamente en {i+1} iteraciones.")
                return beta
        else:
            # Paso fallido: rechazamos y acercamos LM a Descenso de Gradiente
            lam *= 10.0
            
    print(f"LM alcanzó el número máximo de iteraciones ({max_iter}) sin converger a la tolerancia tol={tol}.")
    return beta

In [3]:
# Ajuste de curva exponencial y = b0 * exp(b1 * x)
def exp_func(x, beta):
    return beta[0] * np.exp(beta[1] * x)

def exp_jacobian(x, beta):
    # Derivadas parciales df/d_beta0 y df/d_beta1
    J = np.zeros((len(x), 2))
    J[:, 0] = np.exp(beta[1] * x)                    # df/db0
    J[:, 1] = beta[0] * x * np.exp(beta[1] * x)      # df/db1

    return J

# Generar datos ruidosos para LM
np.random.seed(42)
x_data = np.linspace(0, 4, 50)
beta_true = [2.5, -0.8] # y = 2.5 * exp(-0.8 * x)
y_true = exp_func(x_data, beta_true)
y_noisy = y_true + np.random.normal(0, 0.1, size=len(x_data))

# Punto inicial para LM
beta_guess = [1.0, -0.1]
beta_opt = levenberg_marquardt(x_data, y_noisy, exp_func, exp_jacobian, beta_guess)

print("Parámetros LM verdaderos:", beta_true)
print("Parámetros LM estimados :", np.round(beta_opt, 4))

LM convergió exitosamente en 11 iteraciones.
Parámetros LM verdaderos: [2.5, -0.8]
Parámetros LM estimados : [ 2.5831 -0.8602]
